In [9]:
import itertools
import sqlite3
import pandas as pd
import optuna
# 1. Connect to the database
s = optuna.load_study(study_name="Musical_Instruments_data_MAGCL_tpe_79c3b14b", storage='sqlite:///Musical_Instruments_data/MAGCL.db')


In [10]:
print(s.best_params, s.best_value)

{'lr': 0.001, 'magcl_decay': 0, 'magcl_ssl_weight': 0.1, 'batch_size': 2048} -0.028624


In [12]:
s.trials_dataframe().loc[19]

number                                                                               19
value                                                                               NaN
datetime_start                                               2026-10-14 16:07:47.113362
datetime_complete                                            2026-10-14 16:22:55.465988
duration                                                         0 days 00:15:08.352626
params_batch_size                                                                  2048
params_lr                                                                         0.001
params_magcl_decay                                                                0.001
params_magcl_ssl_weight                                                            0.01
system_attrs_tpe:relative_params:0    {"lr": 0.001, "magcl_decay": 0.001, "magcl_ssl...
state                                                                              FAIL
Name: 19, dtype: object

In [13]:
for t in s.trials:
    if t.state == optuna.trial.TrialState.RUNNING:
        s._storage.set_trial_state_values(t._trial_id,optuna.trial.TrialState.FAIL)
        print('Market Fail',t.number)


Market Fail 18


In [14]:
s.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_lr,params_magcl_decay,params_magcl_ssl_weight,system_attrs_tpe:relative_params:0,state
0,0,-0.026782,2026-10-12 14:17:50.049677,2026-10-12 18:04:39.909375,0 days 03:46:49.859698,2048,0.005,0.00000,0.10,NaN,COMPLETE
1,1,-0.027846,2026-10-12 18:04:39.932098,2026-10-12 21:19:15.591051,0 days 03:14:35.658953,2048,0.005,0.00008,0.01,NaN,COMPLETE
2,2,-0.027122,2026-10-12 21:19:15.619831,2026-10-13 00:26:35.970866,0 days 03:07:20.351035,2048,0.005,0.00100,0.10,NaN,COMPLETE
3,3,-0.026890,2026-10-13 00:26:35.985818,2026-10-13 03:29:15.699437,0 days 03:02:39.713619,2048,0.005,0.00100,0.05,NaN,COMPLETE
4,4,-0.026804,2026-10-13 03:29:15.717052,2026-10-13 06:46:28.498130,0 days 03:17:12.781078,2048,0.005,0.00000,0.05,NaN,COMPLETE
5,5,-0.027839,2026-10-13 06:46:28.541750,2026-10-13 10:00:39.576052,0 days 03:14:11.034302,2048,0.005,0.00008,0.01,NaN,COMPLETE
6,6,-0.028539,2026-10-13 10:00:39.593156,2026-10-13 14:23:07.428138,0 days 04:22:27.834982,2048,0.001,0.00000,0.10,NaN,COMPLETE
7,7,-0.026905,2026-10-13 14:23:07.448892,2026-10-13 17:07:42.918654,0 days 02:44:35.469762,2048,0.005,0.00100,0.05,NaN,COMPLETE
8,8,-0.027841,2026-10-13 17:07:42.931871,2026-10-13 19:41:40.899000,0 days 02:33:57.967129,2048,0.005,0.00008,0.01,NaN,COMPLETE
9,9,-0.026887,2026-10-13 19:41:40.911723,2026-10-13 22:17:05.342108,0 days 02:35:24.430385,2048,0.005,0.00100,0.05,NaN,COMPLETE


In [19]:
PARAM_GRIDS = {
    "CARL": {
        "id_emb_size": [15, 50, 150, 200, 300],
        "filters_num": [50],
        "kernel_size": [3],
        "word_dim": [300],
        "weight_decay": [0.001, 0.005, 0.01, 0.05, 0.1],  # regularization_lambda
        "lr": [0.001],  # learning_rate
        "batch_size": [100, 200, 256],
    },
    "ConvMF": {
        "word_dim": [300],  # see word_dim note below
        "filters_num": [50, 100, 200],
        "kernel_size": [3, 4, 5],
        "fc_dim": [50, 100],
        "drop_out": [0.2, 0.5],
    },
    "DAML": {
        "weight_decay": [0.001, 0.01, 1, 10, 100],  # regularization
        "lr": [0.00001, 0.00005],  # learning_rate
        "id_emb_size": [8, 16, 32],
        "kernel_size": [3],
        "filters_num": [100],
        "word_dim": [300],  # see word_dim note below
    },
    "DeepCoNN": {
        "fc_dim": [25, 50, 100],
        "filters_num": [50, 100, 200],
        "kernel_size": [3],
        "word_dim": [300],  # see word_dim note below
        "drop_out": [0.2, 0.5],
        "weight_decay": [0.001, 0.002, 0.01],  # regularization_lambda
        "batch_size": [100],
    },
    "APH": {
        "aph_dim": [8, 16, 32, 64],
        "lr": [0.0005, 0.001, 0.005],
        "weight_decay": [0.001, 0.01, 0.05, 0.1],
        "batch_size": [128, 256],
    },
    "KANN": {
        "kann_d_model": [50],  # see kann_d_model note below
        "kann_dff": [512, 1024, 2048],
        "kann_num_heads": [2, 4, 8],
        "kann_num_layers": [1, 2, 3],
        "drop_out": [0.1, 0.2, 0.5],
        "batch_size": [64, 128, 256],
    },
    "LETTER": {
        "letter_hidden_dim": [32, 64, 128],
        "drop_out": [0.3],
        "letter_edge_ratio": [1, 5, 10, 20],
        "lr": [0.0005, 0.00001, 0.001],  # learning_rate
        "batch_size": [128, 256],
    },
    "MAGCL": {
        "lr": [0.001, 0.005],
        "magcl_decay": [0, 0.00008, 0.001],
        "magcl_ssl_weight": [0.01, 0.05, 0.1],
        "batch_size": [2048],
    },
    "RLMRec": {
        "rlmrec_kd_weight": [0.001, 0.01, 0.1],
        "rlmrec_kd_temperature": [0.1, 0.2, 0.5],
        "rlmrec_reg_weight": [1e-8, 1e-7, 1e-6],
        "lr": [0.001],
        "batch_size": [4096],
    },
    "LightGCN": {
        "lgcn_layers": [1, 2, 3, 4],
        "lgcn_decay": [0.00001, 0.0001, 0.001, 0.01],
        "lr": [0.0005, 0.001, 0.005],
        "batch_size": [2048],
    },
    "TARMF": {
        "word_dim": [300],  # see word_dim note below
        "id_emb_size": [64, 128, 256],
    },
}

import itertools
grid = PARAM_GRIDS['MAGCL']
done = {tuple (sorted(t.params.items())) for t in s.trials if t.state.name == 'COMPLETE'}
n = 0

for combo in itertools.product(*grid.values()):
    p = dict(zip(grid.keys(), combo))
    if tuple(sorted(p.items())) not in done:
        s.enqueue_trial(p,skip_if_exists=True)
        n+=1
print('enqueued', n, '-> run with --max_trials', len (done)+n)


[I 2026-10-14 16:52:55,574] Trial with params {'lr': 0.001, 'magcl_decay': 0, 'magcl_ssl_weight': 0.01, 'batch_size': 2048} already exists. Skipping enqueue.
[I 2026-10-14 16:52:55,578] Trial with params {'lr': 0.001, 'magcl_decay': 0, 'magcl_ssl_weight': 0.05, 'batch_size': 2048} already exists. Skipping enqueue.
[I 2026-10-14 16:52:55,583] Trial with params {'lr': 0.001, 'magcl_decay': 8e-05, 'magcl_ssl_weight': 0.01, 'batch_size': 2048} already exists. Skipping enqueue.
[I 2026-10-14 16:52:55,587] Trial with params {'lr': 0.001, 'magcl_decay': 8e-05, 'magcl_ssl_weight': 0.05, 'batch_size': 2048} already exists. Skipping enqueue.
[I 2026-10-14 16:52:55,591] Trial with params {'lr': 0.001, 'magcl_decay': 8e-05, 'magcl_ssl_weight': 0.1, 'batch_size': 2048} already exists. Skipping enqueue.
[I 2026-10-14 16:52:55,595] Trial with params {'lr': 0.001, 'magcl_decay': 0.001, 'magcl_ssl_weight': 0.01, 'batch_size': 2048} already exists. Skipping enqueue.
[I 2026-10-14 16:52:55,601] Trial wit

enqueued 12 -> run with --max_trials 18
